# Switching SEC APIs: moving existing Lucene-query code over

If your code already sends Lucene-style queries to an SEC filings API, here's what carries over to Quantisnow as is and what to change: auth headers, paging, sort and field names.

Read it with charts at https://www.quantisnow.com/sec-api/guides/sec-api-alternative. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

Most SEC data APIs built on a search index look alike: a JSON body with a query string, `from`, `size` and `sort`, and a response with a total and a list of records. If yours does, moving over is mostly a base URL and a few field names.

## What works as is

- Lucene query syntax: `field:value`, `AND`, `OR`, `NOT`, parentheses, quoted phrases, ranges like `filedAt:[2026-01-01 TO 2026-03-31]`, comparisons like `purchased.valueUsd:>1000000`, and trailing `*`.
- Paging with `from` and `size`, up to 50 a page and 10,000 a query. [Paging past 10,000](https://www.quantisnow.com/sec-api/guides/paginate-past-10000-results) works by splitting the date range.
- Sort as `[{"filedAt": {"order": "desc"}}]`, or the shorter `"filedAt:desc"`.
- The key in `Authorization: Bearer <key>`, `Authorization: <key>` with no scheme, an `X-API-Key` header, or `?token=` in the URL. Code written for any of those keeps working.

## What to check

- Field names. Ours are listed on each dataset page with what they hold: [filings](https://www.quantisnow.com/sec-api/filings-search), [insider trades](https://www.quantisnow.com/sec-api/insider-trading), [13F](https://www.quantisnow.com/sec-api/13f-holdings), [13D/13G](https://www.quantisnow.com/sec-api/beneficial-ownership), [financials](https://www.quantisnow.com/sec-api/financial-statements), [8-K items](https://www.quantisnow.com/sec-api/8-k-events).
- Endpoints: one per dataset under `https://www.quantisnow.com/api/v1`, plus `/full-text` for searching inside filings.
- Insider trades come one record per filing and insider with purchases and sales already summed in dollars, rather than one record per transaction row.
- Times are US Eastern with an offset, as EDGAR reports acceptance times.

## Before and after

In [ ]:
import os, requests

# The same request shape most SEC search APIs take; only the URL and key change.
res = requests.post(
    "https://www.quantisnow.com/api/v1/filings",
    headers={"Authorization": os.environ["QUANTISNOW_API_KEY"]},
    json={
        "query": 'formType:"10-K" AND ticker:AAPL',
        "from": 0,
        "size": 10,
        "sort": [{"filedAt": {"order": "desc"}}],
    },
)
print(res.json()["total"])

## Try it on your own queries

New accounts get 100 free calls, enough to run a handful of your own queries against us and compare the records before deciding. The [playground](https://www.quantisnow.com/sec-api/playground) shows saved example responses without a key, and the [pricing page](https://www.quantisnow.com/sec-api/pricing) has the plans.

### What has Nvidia filed with the SEC in the last 90 days?

A company's whole filing history for a quarter, newest first, to diff against what you already have.

```shell
curl -X POST https://www.quantisnow.com/api/v1/filings \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "ticker:NVDA AND filedAt:[2026-08-26 TO 2026-11-24]",
  "from": 0,
  "size": 5,
  "sort": [{"filedAt": {"order": "desc"}}]
}'
```

Response (real, first 2 of 25):

```json
{
  "total": {
    "value": 25,
    "relation": "eq"
  },
  "data": [
    {
      "id": "0001696841-26-000016",
      "accessionNo": "0001696841-26-000016",
      "formType": "4",
      "filedAt": "2026-11-23T16:54:28-05:00",
      "cik": "0001045810",
      "companyName": "NVIDIA Corporation",
      "ticker": "NVDA",
      "tickers": [
        "NVDA"
      ],
      "items": [],
      "description": "EVP, General Counsel and Sec Timothy S. Teter sold $4,542,781 worth of shares (30,000 units at $151.43) as part of a pre-agreed trading plan (SEC Form 4)",
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1045810/000169684126000016/0001696841-26-000016-index.htm",
      "quantisnowUrl": "https://www.quantisnow.com/insight/evp-general-counsel-and-sec-timothy-s-teter-sold-4542781-6804297"
    },
    {
      "id": "0001045810-26-000090",
      "accessionNo": "0001045810-26-000090",
      "formType": "8-K",
      "filedAt": "2026-11-23T08:57:54-05:00",
      "cik": "0001045810",
      "companyName": "NVIDIA Corporation",
      "ticker": "NVDA",
      "tickers": [
        "NVDA"
      ],
      "items": [
        "5.02",
        "9.01"
      ],
      "description": "NVIDIA Corporation filed SEC Form 8-K: Leadership Update, Financial Statements and Exhibits",
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1045810/000104581026000090/0001045810-26-000090-index.htm",
      "quantisnowUrl": "https://www.quantisnow.com/insight/nvidia-corporation-filed-sec-form-8-k-leadership-update-financial-statements-6803386"
    }
  ]
}
```